## 实现一个连接Milvus的demo

In [ ]:
# 实例化一个MilvusClient 硬编码
from pymilvus import MilvusClient

# 连接到Milvus服务器的客户端对象
client = MilvusClient("http://192.168.142.128:19530")

# 查看Milvus的版本
print(client.get_server_version())

In [ ]:
client.list_databases()
client.list_collections()

In [ ]:
# 实例化一个MilvusClient
from pymilvus import MilvusClient
from rag_examples.milvus_config import MILVUS_URI

# 连接到Milvus服务器的客户端对象
client1 = MilvusClient(MILVUS_URI, db_name="ai0522")

# 查看Milvus的版本
print(client1.get_server_version())

In [ ]:
# 创建数据库
# 先要把目前有的数据库都列出啦
databases = client1.list_databases()
if "ai0522" not in databases:
    client1.create_database("ai0522")
else:
    print("数据库已存在")

In [ ]:
client1.list_databases()

In [ ]:
client1.list_collections()

In [ ]:
# 创建一个集合（表）
client1.use_database("ai0522")
client1.create_collection(
    collection_name="ai0522_collection",
    dimension=1024,
    auto_id=True)

client1.list_collections()

In [ ]:
# 查看集合的描述, 叫做元数据
client1.describe_collection("ai0522_collection")

In [ ]:
# 创建一个自定义字段的collection
'''
    存的是一篇文章：
    字段列表：
    ID  作为主键  自增长
    title  文章标题  字符串
    content  文章片段内容  字符串
    views   浏览量   整数
    向量字段  向量化  floatvector
    category 分类  字符串
'''
from pymilvus.milvus_client import IndexParams
from pymilvus import FieldSchema, CollectionSchema, DataType
columns = [
    FieldSchema(name="id", dtype=DataType.INT64, is_primary=True, auto_id=True, description="文章ID"),
    FieldSchema(name="title", dtype=DataType.VARCHAR, max_length=100, description="文章标题"),
    FieldSchema(name="content", dtype=DataType.VARCHAR, max_length=65535, description="文章内容"),
    FieldSchema(name="views", dtype=DataType.INT64, description="文章浏览量"),
    FieldSchema(name="vector", dtype=DataType.FLOAT_VECTOR, dim=1024, description="文章向量"),
    FieldSchema(name="category", dtype=DataType.VARCHAR, max_length=100, description="文章分类")
]

# 实例化schema 对象
schema = CollectionSchema(fields=columns, description="文章集合")

# 创建collection
if client1.has_collection("ai0522_docs"):
    client1.drop_collection("ai0522_docs")
client1.create_collection(collection_name="ai0522_docs", schema=schema, metric_type="COSINE")

In [ ]:
documents = [
        {
            "content": "人工智能（AI）是模拟人类智能的计算机科学领域，包括机器学习、深度学习、自然语言处理等技术。",
            "title": "人工智能简介",
            "category": "AI",
            "views": 1000,
            "vector": [0.1232] * 1024
        },
        {
            "content": "机器学习是人工智能的分支，通过训练数据让计算机自动学习规律，无需显式编程。",
            "title": "机器学习基础",
            "category": "AI",
            "views": 800,
            "vector": [0.1232] * 1024
        },
        {
            "content": "深度学习使用多层神经网络模拟人脑，在图像识别、语音识别等领域取得突破性进展。",
            "title": "深度学习入门",
            "category": "AI",
            "views": 1200,
            "vector": [0.1232] * 1024
        },
        {
            "content": "RAG（检索增强生成）结合检索和生成技术，先检索相关知识库，再让大语言模型基于检索结果生成答案。",
            "title": "RAG 技术解析",
            "category": "LLM",
            "views": 600,
            "vector": [0.1232] * 1024
        },
        {
            "content": "Milvus 是一个开源的向量数据库，专门用于存储和搜索向量数据，支持亿级向量毫秒级检索。",
            "title": "Milvus 向量数据库",
            "category": "Database",
            "views": 500,
            "vector": [0.1232] * 1024
        }
    ]

# 把模拟的数据insert到collection中
client1.insert(collection_name="ai0522_docs", data=documents)

# 创建索引
index_param = IndexParams()
index_param.add_index(field_name="vector", index_type="IVF_FLAT", metric_type="COSINE")
client1.create_index(collection_name="ai0522_docs", index_params=index_param)
# 加载集合后才能够进行检索
client1.load_collection(collection_name="ai0522_docs")

In [ ]:
documents = [
        {
            "content": "人工智能（AI）是模拟人类智能的计算机科学领域，包括机器学习、深度学习、自然语言处理等技术。",
            "title": "人工智能简介",
            "category": "AI",
            "views": 1000,
            "vector": [0.1232] * 1024,
            "tags": "学科类"
        },
        {
            "content": "Milvus 是一个开源的向量数据库，专门用于存储和搜索向量数据，支持亿级向量毫秒级检索。",
            "title": "Milvus 向量数据库",
            "category": "Database",
            "views": 500,
            "vector": [0.1232] * 1024,
            "tags": "技术类"
        }
    ]

# 把模拟的数据insert到collection中
client1.insert(collection_name="ai0522_docs", data=documents)

# 创建索引
# index_param = IndexParams()
# index_param.add_index(field_name="vector", index_type="IVF_FLAT", metric_type="COSINE")
# client1.create_index(collection_name="ai0522_docs", index_params=index_param)
# 加载集合后才能够进行检索
client1.load_collection(collection_name="ai0522_docs")

In [ ]:
# 创建一个自定义字段的collection
'''
    存的是一篇文章：
    字段列表：
    ID  作为主键  自增长
    title  文章标题  字符串
    content  文章片段内容  字符串
    views   浏览量   整数
    向量字段  向量化  floatvector
    category 分类  字符串
'''
from pymilvus.milvus_client import IndexParams
from pymilvus import FieldSchema, CollectionSchema, DataType
columns = [
    FieldSchema(name="id", dtype=DataType.INT64, is_primary=True, auto_id=True, description="文章ID"),
    FieldSchema(name="title", dtype=DataType.VARCHAR, max_length=100, description="文章标题"),
    FieldSchema(name="content", dtype=DataType.VARCHAR, max_length=65535, description="文章内容"),
    FieldSchema(name="views", dtype=DataType.INT64, description="文章浏览量"),
    FieldSchema(name="vector", dtype=DataType.FLOAT_VECTOR, dim=1024, description="文章向量"),
    FieldSchema(name="category", dtype=DataType.VARCHAR, max_length=100, description="文章分类")
]

# 实例化schema 对象
schema = CollectionSchema(fields=columns, description="文章集合", enable_dynamic_field=True)

# 创建collection
if client1.has_collection("ai0522_docs_dys"):
    client1.drop_collection("ai0522_docs_dys")
client1.create_collection(collection_name="ai0522_docs_dys", schema=schema)

In [ ]:
documents = [
        {
            "content": "人工智能（AI）是模拟人类智能的计算机科学领域，包括机器学习、深度学习、自然语言处理等技术。",
            "title": "人工智能简介",
            "category": "AI",
            "views": 1000,
            "vector": [0.1232] * 1024,
            "images": "https://www.baidu.com/img/PCtm_d9c8750bed0b3c7d089fa71aee80c7b0.png"
        },
        {
            "content": "Milvus 是一个开源的向量数据库，专门用于存储和搜索向量数据，支持亿级向量毫秒级检索。",
            "title": "Milvus 向量数据库",
            "category": "Database",
            "views": 500,
            "vector": [0.1232] * 1024,
            "tags": "技术类",
            "author": "刘辉",
            "publish": "2025-09-08"
        }
    ]

# 把模拟的数据insert到collection中
client1.insert(collection_name="ai0522_docs_dys", data=documents)

# 创建索引
# index_param = IndexParams()
# index_param.add_index(field_name="vector", index_type="IVF_FLAT", metric_type="COSINE")
# client1.create_index(collection_name="ai0522_docs_dys", index_params=index_param)
# 加载集合后才能够进行检索
client1.load_collection(collection_name="ai0522_docs_dys")

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
load_dotenv()

def generate_embedding_with_llm(texts, model="text-embedding-v4"):
    """使用阿里云百炼 DashScope API 生成真实的 Embedding 向量

    Args:
        texts: 文本列表或单个文本
        model: Embedding 模型名称
        api_key: API Key，默认从环境变量读取

    Returns:
        向量列表（每个向量是浮点数列表）
    """

    api_key = os.getenv("DASHSCOPE_API_KEY") or os.getenv("ALIYUN_API_KEY")
    if not api_key:
        raise ValueError("未找到 API Key，请设置环境变量 DASHSCOPE_API_KEY 或 ALIYUN_API_KEY")

    client = OpenAI(
        api_key=api_key,
        base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    )

    # 判断输入是否是单个文本，还是文本列表
    if isinstance(texts, str):
        texts = [texts]
        single_input = True
    else:
        single_input = False

    try:
        all_embeddings = []
        batch_size = 25

        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]
            response = client.embeddings.create(
                model=model,
                input=batch,
                encoding_format="float",
            )
            batch_embeddings = [item.embedding for item in response.data]
            # print(f"向量后的内容：{response.data}")
            all_embeddings.extend(batch_embeddings)
            print(f"  已处理批次 {i//batch_size + 1}: {len(batch_embeddings)} 条")

        return all_embeddings[0] if single_input else all_embeddings

    except Exception as e:
        print(f"Embedding API 调用失败：{e}")
        print("退化为模拟 Embedding 生成...")
        # return generate_mock_embeddings(texts if not single_input else texts[0])

In [ ]:
generate_embedding_with_llm("人工智能（AI）是模拟人类智能的计算机科学领域")

In [ ]:
documents = [
        {
            "content": "人工智能（AI）是模拟人类智能的计算机科学领域，包括机器学习、深度学习、自然语言处理等技术。",
            "title": "人工智能简介",
            "category": "AI",
            "views": 1000,
            "vector": generate_embedding_with_llm("人工智能（AI）是模拟人类智能的计算机科学领域，包括机器学习、深度学习、自然语言处理等技术。"),
            "images": "https://www.baidu.com/img/PCtm_d9c8750bed0b3c7d089fa71aee80c7b0.png"
        },
        {
            "content": "Milvus 是一个开源的向量数据库，专门用于存储和搜索向量数据，支持亿级向量毫秒级检索。",
            "title": "Milvus 向量数据库",
            "category": "Database",
            "views": 500,
            "vector": generate_embedding_with_llm("Milvus 是一个开源的向量数据库，专门用于存储和搜索向量数据，支持亿级向量毫秒级检索。"),
            "tags": "技术类",
            "author": "刘辉",
            "publish": "2025-09-08"
        }
    ]

# 把模拟的数据insert到collection中
client1.insert(collection_name="ai0522_docs_dys", data=documents)

# 创建索引
# index_param = IndexParams()
# index_param.add_index(field_name="vector", index_type="IVF_FLAT", metric_type="COSINE")
# client1.create_index(collection_name="ai0522_docs_dys", index_params=index_param)
# 加载集合后才能够进行检索
client1.load_collection(collection_name="ai0522_docs_dys")